In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 🧱 LangChat 心智模型 | Week11-Day5

**📌 当前主题：竞品对比 — Dify/LangGraph/OpenClaw/Claude Code 各自的链路，LangChat 最独特的设计是什么？**

**日期：2026-08-14（周五）**

---


## ━━━ 1. 今日核心问题 ━━━

### 为什么 LangChat 不是 Dify？不是 LangGraph？不是 OpenClaw？

过去四周，我在脑子里把 LangChat 的架构一砖一砖搭起来了——四层架构、制品链、FrozenExecutionContext、Capability × Industry 正交模型。但如果你问我"LangChat 和 Dify 有什么区别？"，我大概率会讲一堆术语，然后说"我们更企业级"。

这不是竞品分析，这是自我安慰。

真正的问题不是"我们比别人好"，而是：

**LangChat 在 AI 应用平台这个赛道上，选择了哪条别人没选的路？**

今天我把四个竞品/相关产品放在同一张桌上，用同一条链路（用户意图 → 编排 → 执行 → 治理）拆解，找出 LangChat 的"独选择"。

---


## ━━━ 2. 人话解释（用 Jason 26 年 ERP 经验讲）━━━

Jason，你在 ERP 行业 26 年，一定见过这种场面：

20 年前，有人问"你们的 ERP 和金蝶有什么区别？"销售回答："我们更灵活、更强大、更企业级。"——这种回答等于没回答。

真正的区别要从三个维度看：
1. **制品链**：从"需求"到"上线运行"中间有几个环节？谁在治理每个环节？
2. **执行模型**：运行时是无状态的还是有状态的？执行计划是不可变的还是可修改的？
3. **治理位置**：治理是前置（编译期）还是后置（运行期）？

用 ERP 的话说：
- **Dify** 像"低代码表单工具"——拖拽画布、即时运行、治理靠人。
- **LangGraph** 像"工作流引擎 SDK"——开发者自己编排、自己治理、没有平台层。
- **OpenClaw** 像"运维自动化 Agent"——自己就是执行者、自己定义链路。
- **Claude Code** 像"智能 IDE 插件"——个人开发者工具、不面向企业应用生命周期。

LangChat 选择了第四条路：**制品链治理 + 不可变执行闭包 + 平台化能力供给**。这不是"更好的 Dify"，而是完全不同的物种。

---


## ━━━ 3. LangChat 架构位置 ━━━

今天的内容覆盖整个链路——不是某个站点，而是整条链的设计哲学差异。


In [ ]:
External Clients → [EAC → Blueprint → Compiler → SkillRelease → DeploymentRevision → Runtime] → Enterprise Systems
                   ↑                                                              ↑
                   制品链治理（LangChat 独有）                                      不可变执行闭包（LangChat 独有）


LangChat 的核心独特性不在某一个站点，而在站点之间的**链本身**：
- 每一步都有制品（artifact）
- 每一步都有版本（version）
- 每一步都有 digest（内容寻址）
- 每一步不可跳过、不可逆序

---


## ━━━ 4. ADR 依据 ━━━

### ADR-001：定位锚定

> LangChat 是「企业 AI 应用平台」，不是框架、不是 SDK、不是低代码平台、不是纯 Agent Runtime。

ADR-001 §2 备选方案表显式拒绝了四种定位——"AI 助理"、"RAG 即服务"、"LLM 应用框架"、"AI 中台"。这些定位分别对应不同竞品的核心叙事。

### v2 Charter §3：明确不做

Charter 列了 6 条"不是"：
1. 不是 AI IDE
2. 不是 Workflow Builder
3. 不是低代码平台
4. 不是仅为 Agent Runtime
5. 不保留 WorkflowSpec 在目标态
6. 不是统一业务网关

每一条"不是"都对应一个竞品的"是"。

### Charter §6.2：Single Canonical Execution Path

> 平台只维护一条 canonical 执行路径：从 DeploymentRevision 指向的 digest-pinned SkillRelease，经 Runtime 在 FrozenExecutionContext 中执行。

这是 LangChat 与所有竞品最本质的区别——没有旁路、没有快车道、没有"先跑起来再说"。

### Charter §6.4：Single Artifact Chain


In [ ]:
Blueprint Candidate → BlueprintVersion → ExecutionPlanIR → SkillRelease → DeploymentRevision


五环制品链，不可跳过、不可逆序。这在竞品中绝无仅有。

### ADR-007：三段式架构链

External Clients → Capability Runtime → Enterprise Systems。竞品大多只有中间段（Runtime），缺两侧的治理。

---


## ━━━ 5. 代码验证 ━━━

### 5.1 四个产品的代码结构对比

| 维度 | LangChat | Dify | LangGraph | OpenClaw |
|---|---|---|---|---|
| **包结构** | `business_domain/` `blueprint/` `compiler/` `supply_chain/` `runtime/` `skill_release/` | `api/` `services/` `controllers/` | `langgraph/` (单一 Python 包) | `openclaw/` (CLI + Agent Loop) |
| **核心抽象** | 制品链（5环不可变制品） | Workflow + App | StateGraph + Node | Agent + Tool + Session |
| **制品概念** | ✅ BlueprintVersion/ExecutionPlanIR/SkillRelease/DeploymentRevision 各有 digest | ❌ 只有 App 版本号 | ❌ 只有代码版本 | ❌ 有 Skill 但不是不可变制品 |
| **执行入口** | `runtime/canonical_entry.py` → `execute()` | `api/app.py` 直接调 LLM | `graph.invoke(state)` | `sessions_send` / `sessions_spawn` |
| **治理嵌入** | 制品链每环携带 effect_policy / required_scopes | 运行时 check | 开发者自理 | 平台层 session 级控制 |

### 5.2 关键代码签名对比

**LangChat 的执行入口（`runtime/canonical_entry.py`）：**


In [ ]:
def execute(
    deployment_revision: DeploymentRevision,
    frozen_context: FrozenExecutionContextV2,
    *,
    runtime_factory: RuntimeFactory,
    kb_search_fn: KbSearchFn,
    llm_chat_fn: LlmChatFn,
    tool_call_fn: ToolCallFn,
) -> ExecutionResult:


执行前必须传入 `DeploymentRevision`（digest-pinned 闭包）和 `FrozenExecutionContext`（不可变上下文）。没有这两个对象，执行不会发生。

**Dify 的执行路径（概念性）：**


In [ ]:
# Dify: WorkflowRunner 直接消费 Workflow 配置
workflow_runner.run(workflow, inputs, user_id)


配置直接变执行。没有"编译"步骤、没有"制品"步骤、没有"部署闭包"。

**LangGraph 的执行路径：**


In [ ]:
# LangGraph: 图编译后直接执行
app = graph.compile()
result = app.invoke(initial_state)


开发者自己定义节点、自己连线、自己执行。没有平台层治理。

**OpenClaw 的执行路径：**


In [ ]:
# OpenClaw: Agent loop 直接调用工具
# session_send → LLM → tool_call → result


OpenClaw 是 Agent Host，不是 AI 应用平台。它调用 LangChat 的 Capability，自己不拥有制品链。

### 5.3 LangChat 独有结构


In [ ]:
# business_domain/__init__.py — 目标态业务对象（不可变）
class ApplicationContract: ...     # 业务 API 契约
class DigitalEmployeeDefinition: ... # 数字员工定义

# blueprint/__init__.py — 源制品链
class BlueprintCandidate: ...      # 候选源制品
class BlueprintVersion: ...        # canonical 源制品（不可变）

# compiler/core.py — 确定性编译
def compile(snapshot: SemanticCompilerInputSnapshot) -> ExecutionPlanContent: ...

# supply_chain/__init__.py — 制品供应链
class Build: ...                   # 构建任务
class ExecutionPlanIR: ...         # 中间表示（不可编辑）
class CompatibilityMatrix: ...     # 运行时兼容矩阵

# runtime/canonical_entry.py — 唯一执行入口
def execute(deployment_revision, frozen_context, ...): ...

# runtime/frozen_execution_context.py — 不可变执行闭包
class FrozenExecutionContextV2: ... # 身份+授权+制品 digest+Trace


这条链在 Dify/LangGraph/OpenClaw/Claude Code 中**完全不存在**。这不是功能差异，是架构范式差异。

---


## ━━━ 6. 商业地产映射（LangChat → MI CRE 场景）━━━

用 Jason 最熟悉的商业地产（Commercial Real Estate）场景，看四个产品分别会怎么做"租户合同异常分析"：

| 场景步骤 | LangChat | Dify | LangGraph | OpenClaw |
|---|---|---|---|---|
| **定义能力** | `mall.operation.collection.query` Capability Contract，声明 effect_policy=read_only | 在 App 编辑器里加一个 HTTP 节点 | 开发者写一个 Python Node | 在 Skill 中定义一个 tool |
| **编排逻辑** | Blueprint → Compiler → ExecutionPlanIR（确定性编译） | 拖拽画布连节点 | 写 StateGraph 代码 | Agent loop 自主决策 |
| **部署上线** | SkillRelease → DeploymentRevision（digest-pinned），回滚 = 切 revision | 发布 App 版本 | 部署代码 | 更新 Skill |
| **执行时** | FrozenExecutionContext 闭包执行，知识快照锁定 | 直接调 LLM + API | 直接 invoke graph | Agent 调 tool |
| **审计追溯** | "2026-08-14 10:23 的分析引用了哪个版本的合同政策知识？" → DeploymentRevision digest 精确回答 | "用的是当时的 App 版本" → 粗粒度 | 靠 git log | 靠 session log |
| **知识版本** | KnowledgeSnapshot 被 DeploymentRevision 闭包锁定 | 知识库即文件，随时改 | 开发者自理 | 无知识治理 |
| **多租户隔离** | tenant/workspace + 六维权限，治理嵌入制品 | App 级别隔离 | 无 | Session 级别 |

**MI CRE 管理者的视角：**

如果租户投诉"上个月你们 AI 说我欠租 50 万，但这个月说不欠了"，LangChat 能精确回答"那个回答用的知识快照版本是 KS-2026-07-15-a3f，其中合同条款 §3.2 与最新版本不同"。Dify 做不到——因为它不知道"上个月的回答"基于哪个版本的文件。

---


## ━━━ 7. 与传统方案比较 ━━━

### 7.1 四维对比矩阵

| 维度 | LangChat | Dify | LangGraph | OpenClaw / Claude Code |
|---|---|---|---|---|
| **核心叙事** | 企业 AI 应用平台 | 开源 LLMOps 平台 | Agent 编排框架 | Agent Host / 开发者工具 |
| **目标用户** | 企业架构团队（平台 + 应用双角色） | AI 应用开发者 / 小团队 | 后端工程师 | 个人助手 / 开发者 |
| **制品链** | 5 环不可变制品（Blueprint → IR → SkillRelease → DeploymentRevision） | 无制品概念，App 配置即全部 | 代码即制品（git 管理） | Skill 定义即制品 |
| **编译步骤** | 确定性编译（BlueprintVersion → ExecutionPlanIR） | 无编译，画布配置直接执行 | 图编译（compile），但不产出不可变制品 | 无 |
| **执行模型** | 不可变闭包执行（FrozenExecutionContext） | 可变运行时（配置随时改） | 可变状态图（state 可被 node 修改） | Agent loop（每轮重新决策） |
| **治理位置** | 前置（编译期嵌入 effect_policy / required_scopes） | 后置（运行时权限检查） | 开发者嵌入代码 | 平台层注入 |
| **知识治理** | KnowledgeCollection + KnowledgeSnapshot（目标态） | 文件管理 + 向量检索 | 开发者自己接 RAG | 无 |
| **多租户** | 原生多租户 + 六维权限 | App 级别 | 无 | Session 级别 |
| **可审计性** | 制品 digest + FEC + Trace 全链路 | App 版本 + 运行日志 | git log + 运行日志 | Session transcript |
| **部署可复现** | ✅（同一 Build 输入 = 同一 IR digest） | ❌（配置可随时改） | 部分（代码可复现，但依赖不在闭包内） | ❌ |

### 7.2 各产品的"最佳适用场景"

| 产品 | 最佳场景 | 不适合的场景 |
|---|---|---|
| **LangChat** | 中大型企业需要治理 + 审计 + 多租户的 AI 应用 | 个人开发者快速原型 / 单人助手 |
| **Dify** | 团队快速搭建 AI 应用 / PoC / 不需要严格治理 | 需要审计追溯、知识版本化、多租户隔离 |
| **LangGraph** | 后端工程师需要精细控制 Agent 编排逻辑 | 需要平台化能力（多租户、制品治理、知识管理） |
| **OpenClaw** | Agent Host 场景（调用 LangChat Capability） | 不适合作为 AI 应用平台（它不是） |
| **Claude Code** | 开发者编程助手 | 企业 AI 应用（不是同赛道） |

### 7.3 为什么 LangChat 选择了最难的路？

LangChat 完全可以像 Dify 一样——画布编排、即时运行、快速上线。为什么要搞五环制品链 + 确定性编译 + 不可变执行闭包？

因为**企业场景的不可妥协项不是速度，是确定性**。

- "这个 AI 做的决策基于什么版本的知识？" — 必须能回答
- "回滚到上周的版本" — 必须精确到 digest
- "三个租户同时用不同版本的能力" — 必须互不干扰
- "合规审计要追查 3 个月前的某次执行" — 必须完整可追溯

这些需求在 Dify/LangGraph/OpenClaw 中要么做不到，要么靠"工程努力"硬补。LangChat 把它们变成架构内建。

---


## ━━━ 8. 架构师思考题 ━━━

### 思考题（CTO 级）

**场景：** 你的客户是某大型商业地产集团，旗下 20 个购物中心，每个项目公司有自己的 IT 系统（SAP / Oracle / 金蝶混用）。客户要建"集团 AI 智能助手"，需求是：

1. 集团总部统一管控 AI 能力版本
2. 每个项目公司可以自定义工作流
3. 所有 AI 回答可追溯到当时的知识版本
4. 合规部门要求审计日志精确到"这次回答用了哪个版本的哪份合同"

**问题：**
1. 你会选 LangChat 还是 Dify？为什么？
2. 如果选 LangChat，20 个项目公司的"自定义工作流"放在制品链的哪一环？是每个项目公司一个 BlueprintVersion，还是一个 BlueprintVersion + N 个 DeploymentRevision？
3. 如果项目公司 A 的合同政策更新了，但项目公司 B 没更新，DeploymentRevision 闭包怎么保证两个项目公司用不同的 KnowledgeSnapshot？
4. OpenClaw 在这个场景中扮演什么角色？它和 LangChat 的边界在哪？

### 提示

- 回到 Charter §6.6：DeploymentRevision 是完整运行时闭包
- 回到 ADR-003：Capability × Industry 正交模型
- 回到 Week 8 学的链路：Agent Host 调用 LangChat，LangChat 不拥有 Agent Host

---


## ━━━ 9. 我的理解变化 ━━━

**以前以为：** LangChat 和 Dify 是同一赛道的竞品，功能列表重合度很高，区别只是"企业级 vs 开源社区"。

**现在知道：** 它们根本不在同一条赛道上。Dify 解决的是"AI 应用怎么快速搭建和运行"的问题——它的核心价值是**速度**。LangChat 解决的是"AI 应用怎么在企业治理框架下确定性运行"的问题——它的核心价值是**确定性**。

这不是"更好的 Dify"，而是"Dify 范式的反面"。

Dify 说"配置即应用，改了就生效"。
LangChat 说"制品即应用，改了要走完整链路"。

这两句话背后是完全不同的架构哲学、完全不同的目标客户、完全不同的商业模型。

同理，LangGraph 不是竞品——它是**可以被 LangChat 封装在 SkillRelease 里的编排能力**。OpenClaw 不是竞品——它是**调用 LangChat 的 Agent Host**。Claude Code 不是竞品——它根本不在企业 AI 应用平台赛道上。

LangChat 最独特的设计是：**把软件工程的最佳实践（制品链、确定性构建、不可变部署）引入 AI 应用治理**。这不是功能创新，是范式创新。

---


## ━━━ 10. 明日连接 + Semantic Layer ━━━

### 明日预告（Day 6 周六）

**⚡ 实战交付：LangChat v2 实施路线图 v1.0 — 按 Sprint 拆分**

过去两周（Day 1-Day 5），我做完了三件事：
- Capability Inventory（能力清查）
- Gap Matrix（差距矩阵）
- 竞品对比（今天）

明天把这三件事的输出综合成一份**可执行的实施路线图**：
- 前 3 个 Sprint 做什么
- 每个 Sprint 的验收标准
- 风险与依赖

这将是 Week 11 最重要的交付物——它把 4 周的心智模型建设转化为行动方案。

### Semantic Layer 位置

今天的知识在整个语义层级中的位置：


In [ ]:
Ontology（存在论）
  └─ "AI 应用平台"这个概念有哪些可能形态？
     └─ Domain Model（领域模型）
        └─ LangChat 选择了"制品链治理"范式
           └─ Capability（能力）
              └─ "对比分析"是一种架构师能力，不是代码能力
                 └─ Skill（技能）
                    └─ 今天产出的是"竞品定位判断框架"


四周学习即将结束。这条从 Ontology 到 Skill 的认知链，已经建成。

---

> 📖 **Engineering Journal 条目同步追加至** `/root/learning-notebooks/engineering-journal.md`
